# GUÍA PRÁCTICA PARA EXAMEN - Aprendizaje No Supervisado

**Objetivo:** Entender y practicar los conceptos clave para el examen.

**Temas a cubrir:**
1. Cargar y preparar imágenes
2. PCA (reducción de dimensiones)
3. K-Means (clustering)
4. Evaluar clustering (silhouette, elbow method)
5. Semi-supervisado (LabelSpreading)
6. Aprendizaje activo

---

## ¿Qué es el Aprendizaje No Supervisado?

**Definición simple:** El modelo **descubre patrones** en los datos SIN etiquetas.

**Ejemplo:** Si tienes 1000 fotos de caras y NO sabes quién es cada persona, el modelo puede agruparlas por similitud.

**Diferencia con supervisado:**
- **Supervisado:** Tienes datos + etiquetas → el modelo aprende a predecir
- **No supervisado:** Tienes SOLO datos → el modelo descubre grupos

---

## BLOQUE 1: Conceptos Básicos

### ¿Qué es K-Means?

K-Means es un algoritmo que **agrupa datos en k clusters** (grupos).

**Algoritmo paso a paso:**
1. Elige k centroides aleatorios
2. Asigna cada dato al centroide más cercano
3. Recalcula los centroides como el promedio de cada grupo
4. Repite 2-3 hasta que converja

**Ventajas:**
- Rápido
- Fácil de entender
- Funciona bien con imágenes

**Desventajas:**
- Necesitas elegir k (número de clusters)
- Puede quedarse en óptimos locales

---

### ¿Qué es PCA?

PCA = **Principal Component Analysis** = Reducción de dimensiones

**Problema:** Una imagen 64×64 = 4096 dimensiones. Demasiadas para procesar.

**Solución:** PCA busca las dimensiones más importantes y reduce la dimensión a 50.

**Analogía:** Si tienes 100 variables, PCA encontrará las 10 más importantes que explican el 95% de la varianza.

---

### ¿Qué es Silhouette Score?

Métrica para evaluar si el clustering es **bueno**.

- **Rango:** -1 a 1
- **Cercano a 1:** Clustering excelente (grupos bien separados)
- **Cercano a 0:** Clustering mediocre (grupos solapados)
- **Cercano a -1:** Clustering malo (datos asignados a grupo equivocado)

---

In [ ]:
# EJERCICIO 1: Cargar imágenes desde carpeta
# Objetivo: Entender cómo se cargan y transforman imágenes

from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# ======================================
# PASO 1: Define la ruta donde están las imágenes
# ======================================
data_root = Path(r'C:\Users\elmer\Documents\archive\all unlabeled')

# Busca todas las imágenes
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_paths = sorted([p for p in data_root.rglob('*') if p.suffix.lower() in image_extensions])

print(f'✓ Imágenes encontradas: {len(image_paths)}')
print(f'\nPrimeras 3 rutas:')
for path in image_paths[:3]:
    print(f'  - {path.name}')

In [ ]:
# EJERCICIO 2: Función para cargar imágenes
# Objetivo: Convertir imágenes a arrays de números

def cargar_imagenes(rutas, tamaño=(64, 64), max_imagenes=None):
    """
    Carga imágenes y las convierte a escala de grises.
    
    Args:
        rutas: Lista de rutas a imágenes
        tamaño: Tamaño para redimensionar (ancho, alto)
        max_imagenes: Máximo número de imágenes a cargar (None = todas)
    
    Returns:
        images: Array de imágenes (n_imagenes, alto, ancho)
        X: Array aplanado (n_imagenes, alto*ancho)
    """
    rutas_seleccionadas = rutas if max_imagenes is None else rutas[:max_imagenes]
    
    imagenes_array = []
    for ruta in rutas_seleccionadas:
        with Image.open(ruta) as img:
            # Convierte a escala de grises
            img_gris = img.convert('L')
            # Redimensiona
            img_redim = img_gris.resize(tamaño)
            # Convierte a array y normaliza (0-1)
            arr = np.asarray(img_redim, dtype=np.float32) / 255.0
            imagenes_array.append(arr)
    
    # Stack: junta todos los arrays
    images = np.stack(imagenes_array)
    # Aplanar para que sea (n_imagenes, num_pixeles)
    X = images.reshape(len(images), -1)
    
    return images, X

# Carga las primeras 500 imágenes como prueba
print('Cargando primeras 500 imágenes...')
images, X = cargar_imagenes(image_paths, tamaño=(64, 64), max_imagenes=500)

print(f'\n✓ Carga exitosa:')
print(f'  - images.shape = {images.shape}  (500 imágenes de 64×64)')
print(f'  - X.shape = {X.shape}  (500 imágenes aplanadas a 4096 números)')

In [ ]:
# EJERCICIO 3: Ver algunas imágenes
# Objetivo: Verificar que se cargaron correctamente

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.reshape(-1)

for i in range(10):
    axes[i].imshow(images[i], cmap='gray')
    axes[i].set_title(f'Imagen {i}', fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print('✓ Las imágenes se cargaron correctamente')

---

## BLOQUE 2: Reducción de Dimensiones (PCA)

**Problema:** 500 imágenes × 4096 píxeles = MUCHO cálculo

**Solución:** Usar PCA para reducir a 50 dimensiones importantes

**¿Por qué funciona?** Porque muchos píxeles son redundantes (ej: fondo blanco).

---

In [ ]:
# EJERCICIO 4: Aplicar PCA
# Objetivo: Reducir dimensiones de 4096 → 50

from sklearn.decomposition import PCA

# Crea modelo PCA con 50 componentes
pca = PCA(n_components=50, random_state=42)

# Entrena y transforma
X_reducido = pca.fit_transform(X)

print(f'✓ PCA aplicado:')
print(f'  - Antes: X.shape = {X.shape}')
print(f'  - Después: X_reducido.shape = {X_reducido.shape}')
print(f'\n✓ Varianza explicada:')
varianza_total = pca.explained_variance_ratio_.sum()
print(f'  - 50 componentes explican el {varianza_total*100:.1f}% de la varianza')
print(f'  - Significa: No perdemos casi información, pero calculamos mucho más rápido')

---

## BLOQUE 3: K-Means Clustering

Ahora que tenemos datos reducidos, aplicamos K-Means.

**Preguntas importantes:**
- ¿Cuántos clusters (k) debo elegir?
- ¿Cómo sé si el resultado es bueno?

**Respuestas:**
- Prueba varios valores de k (2, 3, 4, 5, ...)
- Usa **elbow method** o **silhouette score**

---

In [ ]:
# EJERCICIO 5: Encontrar el mejor k usando Elbow Method
# Objetivo: Decidir cuántos clusters usar

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Prueba diferentes valores de k
k_valores = range(2, 11)  # Prueba k de 2 a 10
inertias = []
silhouettes = []

print('Probando diferentes valores de k...')
for k in k_valores:
    # Crea y entrena K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = kmeans.fit_predict(X_reducido)
    
    # Calcula inercia (qué tan apretados están los clusters)
    inertias.append(kmeans.inertia_)
    
    # Calcula silhouette score (qué tan bien se separan)
    sil = silhouette_score(X_reducido, etiquetas)
    silhouettes.append(sil)
    
    print(f'  k={k}: inercia={kmeans.inertia_:.0f}, silhouette={sil:.4f}')

# Encuentra el mejor k por silhouette
mejor_k = k_valores[np.argmax(silhouettes)]
print(f'\n✓ Mejor k por silhouette: {mejor_k}')

In [ ]:
# EJERCICIO 6: Visualizar Elbow Method y Silhouette Score
# Objetivo: Entender por qué elegimos ese k

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico 1: Inercia (elbow method)
axes[0].plot(k_valores, inertias, 'o-', linewidth=2, markersize=8, color='tab:blue')
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].set_xlabel('k (número de clusters)', fontsize=11)
axes[0].set_ylabel('Inercia', fontsize=11)
axes[0].grid(alpha=0.3)
axes[0].axvline(mejor_k, color='red', linestyle='--', label=f'Mejor k={mejor_k}')
axes[0].legend()

# Gráfico 2: Silhouette Score
axes[1].plot(k_valores, silhouettes, 's-', linewidth=2, markersize=8, color='tab:green')
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_xlabel('k (número de clusters)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].grid(alpha=0.3)
axes[1].axvline(mejor_k, color='red', linestyle='--', label=f'Mejor k={mejor_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print('✓ El codo del gráfico izquierdo muestra dónde la mejora se estabiliza')
print('✓ El pico del gráfico derecho muestra el mejor silhouette score')

In [ ]:
# EJERCICIO 7: Entrenar K-Means con el mejor k
# Objetivo: Obtener clusters finales

# Entrena K-Means con el mejor k
kmeans_final = KMeans(n_clusters=mejor_k, random_state=42, n_init=10)
etiquetas_cluster = kmeans_final.fit_predict(X_reducido)

# Cuenta cuántas imágenes en cada cluster
unique, counts = np.unique(etiquetas_cluster, return_counts=True)

print(f'✓ K-Means entrenado con k={mejor_k}:')
for cluster_id, count in zip(unique, counts):
    print(f'  - Cluster {cluster_id}: {count} imágenes ({count/len(etiquetas_cluster)*100:.1f}%)')

In [ ]:
# EJERCICIO 8: Visualizar clusters en 2D (PCA)
# Objetivo: Ver cómo se separan visualmente los clusters

# Reduce a 2D para visualizar
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_reducido)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=etiquetas_cluster, cmap='tab10', s=30, alpha=0.6)
plt.scatter(pca_2d.transform(kmeans_final.cluster_centers_)[:, 0], 
           pca_2d.transform(kmeans_final.cluster_centers_)[:, 1],
           c='black', marker='X', s=300, edgecolors='white', linewidths=2, label='Centroides')
plt.title(f'Clusters visualizados en 2D (k={mejor_k})', fontsize=12, fontweight='bold')
plt.xlabel('Componente PCA 1', fontsize=11)
plt.ylabel('Componente PCA 2', fontsize=11)
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('✓ Colores diferentes = clusters diferentes')
print('✓ Xs negras = centroides (centros de los clusters)')

---

## BLOQUE 4: Checklist para el Examen

### Lo que DEFINITIVAMENTE puede caer:

**1. Cargar un dataset de imágenes**
- [ ] Usar `Path` para buscar archivos
- [ ] Convertir a escala de grises con `PIL.Image.convert('L')`
- [ ] Normalizar a [0, 1]

**2. Aplicar PCA**
- [ ] Entender que reduce dimensiones
- [ ] Usar `PCA(n_components=50)`
- [ ] Saber que explica ~80% de varianza

**3. K-Means Clustering**
- [ ] Entender el algoritmo paso a paso
- [ ] Saber que necesitas elegir k
- [ ] Usar `KMeans(n_clusters=k)`

**4. Evaluar Clustering**
- [ ] Elbow method: gráfico de inercia
- [ ] Silhouette score: -1 a 1
- [ ] Elegir k que maximiza silhouette

**5. Semi-supervisado (LabelSpreading)**
- [ ] Usar con etiquetas parciales
- [ ] Propagar etiquetas a datos no etiquetados

**6. Aprendizaje Activo**
- [ ] Identificar datos más inciertos
- [ ] Seleccionar para etiquetar manualmente

---

## BLOQUE 5: Tips para Modificar Código (Lo que probablemente te pida el profesor)

### Cambio 1: Usar distinto tamaño de imagen

**Problema actual:** `tamaño=(64, 64)`

**Cambio:** `tamaño=(128, 128)` o `(32, 32)`

```python
images, X = cargar_imagenes(image_paths, tamaño=(128, 128), max_imagenes=500)
```

---

### Cambio 2: Usar diferente número de componentes PCA

**Problema actual:** `n_components=50`

**Cambio:** `n_components=100` o `n_components=20`

```python
pca = PCA(n_components=100, random_state=42)
X_reducido = pca.fit_transform(X)
```

---

### Cambio 3: Cambiar k manualmente

**Problema actual:** `k=2` (elegido automáticamente)

**Cambio:** Usar específicamente `k=4` o `k=3`

```python
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)  # k=4 específicamente
etiquetas = kmeans.fit_predict(X_reducido)
```

---

### Cambio 4: Cambiar a RGB (en lugar de escala de grises)

**Problema actual:** `img.convert('L')` (escala de grises)

**Cambio:** `img.convert('RGB')` o mantener RGB original

```python
img_color = img.convert('RGB')  # En lugar de 'L'
```

---

### Cambio 5: Cargar solo un subset de imágenes

**Problema actual:** Cargar todas o primeras 500

**Cambio:** Cargar imágenes específicas

```python
images, X = cargar_imagenes(image_paths, tamaño=(64, 64), max_imagenes=1000)  # 1000 en lugar de 500
```

---

## PLAN DE ESTUDIO (3 SEMANAS)

### SEMANA 1: Conceptos Básicos
- [ ] Entender qué es aprendizaje no supervisado
- [ ] Estudiar K-Means (algoritmo paso a paso)
- [ ] Estudiar PCA (qué reduce y por qué)
- [ ] Hacer ejercicios 1-4 de este notebook
- [ ] **Objetivo:** Poder cargar imágenes y aplicar PCA sin mirar documentación

### SEMANA 2: Clustering y Evaluación
- [ ] Estudiar Elbow Method
- [ ] Estudiar Silhouette Score
- [ ] Hacer ejercicios 5-8 de este notebook
- [ ] Practicar con diferentes valores de k
- [ ] **Objetivo:** Poder elegir k automáticamente usando silhouette

### SEMANA 3: Semi-supervisado y Examen
- [ ] Revisar LabelSpreading
- [ ] Revisar Aprendizaje Activo
- [ ] Practicar modificar el código (cambiar tamaños, k, etc.)
- [ ] Resolver problemas similares a lo que hicimos en clase
- [ ] **Objetivo:** Estar listo para el examen

---

## Preguntas Típicas del Examen (con respuestas)

### P1: ¿Cuál es la diferencia entre aprendizaje supervisado y no supervisado?

**Respuesta:**
- **Supervisado:** Tienes datos + etiquetas reales. El modelo aprende a predecir.
- **No supervisado:** Tienes SOLO datos. El modelo descubre patrones/grupos.

---

### P2: ¿Qué hace K-Means?

**Respuesta:**
K-Means agrupa datos en k clusters buscando minimizar la distancia entre cada dato y el centroide de su cluster.

---

### P3: ¿Cómo se elige el número de clusters (k)?

**Respuesta:**
Usando Elbow Method o Silhouette Score. Se prueba k=2, 3, 4, 5... y se elige el que maximiza silhouette o donde se ve el "codo" en la curva de inercia.

---

### P4: ¿Por qué aplicar PCA antes de K-Means?

**Respuesta:**
Porque reduce dimensiones (4096 → 50) haciendo más rápido el cálculo sin perder información importante.

---

### P5: ¿Qué significa un silhouette score de 0.2?

**Respuesta:**
Que el clustering es mediocre. Los clusters están poco separados. Un valor cercano a 1 sería excelente.

---